In [1]:
import librosa
import numpy as np
import os
from skimage.feature import local_binary_pattern


def extract_mfcc(y, sr, n_mfcc=80, n_fft=512, hop_length=0.01, win_length=0.025, n_mels=128):
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=int(win_length * sr),
        hop_length=int(hop_length * sr),
        n_mels=n_mels,
        win_length=int(win_length * sr)
    )

    #if mfcc.shape[1] < 9: return None

    # delta = librosa.feature.delta(mfcc)

    # # delta-delta
    # delta2 = librosa.feature.delta(mfcc, order=2)

    # # ghép feature
    # feat_all = np.vstack([mfcc, delta, delta2]).T
    # CMVN
    return mfcc.T   # (num_frames, n_mfcc)


In [2]:
import numpy as np

def clbp(image, P=8, R=1):
    """
    image : 2D array (MFCC / spectrogram)
    P : số neighbor
    R : radius
    return:
        CLBP_S, CLBP_M, CLBP_C
    """
    h, w = image.shape

    # padding
    pad = R
    img = np.pad(image, pad, mode='edge')

    # center
    center = img[pad:pad+h, pad:pad+w]

    # init
    CLBP_S = np.zeros((h, w), dtype=np.uint8)
    CLBP_M = np.zeros((h, w), dtype=np.uint8)

    # magnitude threshold
    diff_list = []

    for p in range(P):
        theta = 2 * np.pi * p / P
        dy = int(round(R * np.sin(theta)))
        dx = int(round(R * np.cos(theta)))

        neighbor = img[pad+dy:pad+dy+h, pad+dx:pad+dx+w]

        diff = neighbor - center
        diff_list.append(np.abs(diff))

    # threshold cho magnitude
    diff_stack = np.stack(diff_list, axis=0)
    T = diff_stack.mean()

    for p in range(P):
        theta = 2 * np.pi * p / P
        dy = int(round(R * np.sin(theta)))
        dx = int(round(R * np.cos(theta)))

        neighbor = img[pad+dy:pad+dy+h, pad+dx:pad+dx+w]
        diff = neighbor - center

        # CLBP_S
        CLBP_S |= ((diff >= 0).astype(np.uint8) << p)

        # CLBP_M
        CLBP_M |= ((np.abs(diff) >= T).astype(np.uint8) << p)

    # CLBP_C
    CLBP_C = (center >= center.mean()).astype(np.uint8)

    return CLBP_S, CLBP_M, CLBP_C

def hist(x, bins=256):
    h, _ = np.histogram(x.ravel(), bins=bins, range=(0, bins))
    return h



In [ ]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern

def build_mfcc_dataset(
    wav_list,
    sr=16000,
    segment_sec=2,
    out_path="X_mfcc.npy"
):
    all_feats = []

    segment_len = int(segment_sec * sr)
    count = 1
    for wav_path in wav_list:
        print("Processing:", count)
        count += 1
        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:
            mfcc = extract_mfcc(segment, sr)
            
            if mfcc is None: continue
            CLBP_S, CLBP_M, CLBP_C = clbp(mfcc, P=8, R=1)
            h_s = hist(CLBP_S)
            h_m = hist(CLBP_M)
            h_c = hist(CLBP_C, bins=2)

            feature = np.concatenate([h_s, h_m, h_c])
            all_feats.append(feature)

        #print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [ ]:
import os
CLASS = "strong"

DIR = r"E:\Pythonfile\Voice-Activity-Detect\data\raw\freesound_background_noise"

wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".mp3")
]
print(wav_list)
X_mfcc = build_mfcc_dataset(
    wav_list,
    out_path=f"E:/Pythonfile/Voice-Activity-Detect/data/feature/train/mfcc_clbp_nonspeech_80_nodelta_vivo.npy"
)

['E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108191_Lisola_ok.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108194_Isola Soundwalk Alberto.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108195_alesozziworkshop.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108196_carnevali sound walk.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108197_federico_cortesi.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108198_field-recording-291010mi.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\raw\\freesound_background_noise\\Acoustic environment_108199_massobrio-orsi_workshp_edit_finale.mp3.mp3', 'E:\\Pythonfile\\Voice-Activity